# Project Title
Nasa Near-Earth Asteroids Analysis and Hazardous Classification

### Project Goal
To analyze nasa data on near-Earth asteroids and build a machine learning model that can predict whether an asteroid is potentially hazardous to Earth.

### Main Objective
1. Explore and understand the characteristics of Near-Earth Asteroids
2. Identify patterns related to potentially hazardous asteroids (is_pha)
3. Prepare the dataset for machine learning modeling
4. Build and evaluate a classification model
5. Communicate key findings

### Dataset:
NASA Near-Earth Asteroids & Close Approaches (from Kaggle, originally from NASA JPL)

### Expected Analytical Steps:
1. Data Collection
2. Data Loading & Initial Inspection
3. Data Cleaning & Wrangling
4. Exploratory Data Analysis (EDA)
5. Feature Engineering
6. Modeling (Classification)
7. Evaluation & Conclusions

### Potential Challenges:
- Many columns with missing values
- Imbalanced classes (few hazardous asteroids)
- Understanding astronomical terms

### Success Criteria:
- Clean, well-documented dataset
- Working classification model with reasonable accuracy
- Clear visualizations and insights
- Professional Jupyter notebook and GitHub repo

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load in and inspect the data
df = pd.read_csv('near_earth_asteroids_2025.csv', low_memory=False)
#print(df.head(10))
df.isnull().sum().sort_values(ascending=False)

name                     41099
albedo                   40077
rot_per                  39100
data_arc                   406
data_arc_years             406
moid_km                    131
moid_au                    131
moid_lunar_distances       131
condition_code               2
H                            2
first_obs                    1
pha                          0
spkid                        0
class                        0
size_category                0
diameter_m                   0
diameter_is_estimated        0
diameter_km                  0
full_name                    0
pdes                         0
e                            0
per_y                        0
per                          0
ad                           0
q                            0
a                            0
i                            0
n                            0
last_obs                     0
dtype: int64

In [3]:
# I will be dropping the name, albedo, and the rot_per columns as more than 90% of the data is missing.
df = df.drop(columns=['name', 'albedo', 'rot_per'])
print(df.isnull().sum().sort_values(ascending=False))

data_arc                 406
data_arc_years           406
moid_km                  131
moid_au                  131
moid_lunar_distances     131
condition_code             2
H                          2
first_obs                  1
full_name                  0
spkid                      0
class                      0
size_category              0
diameter_is_estimated      0
diameter_m                 0
diameter_km                0
pha                        0
pdes                       0
e                          0
per_y                      0
per                        0
ad                         0
q                          0
a                          0
i                          0
n                          0
last_obs                   0
dtype: int64


In [4]:
# Here, I'm dropping the rest of the rows that have atleast one missing value. 
df = df.dropna()
print('New shape:', df.shape)
print('Missing values left:', df.isnull().sum().sum())

New shape: (40742, 26)
Missing values left: 0


In [5]:
# Checking for imbalance in the target variable
print(df['pha'].value_counts())
print(df['pha'].value_counts(normalize=True).round(4) * 100)

pha
False    38203
True      2539
Name: count, dtype: int64
pha
False    93.77
True      6.23
Name: proportion, dtype: float64


In [7]:
# Checking for impossible values
print('Diameter (km) min/max:')
print(df['diameter_km'].min(), df['diameter_km'].max())

print('\nMOID (AU) min/max:')
print(df['moid_au'].min(), df['moid_au'].max())

print('\nEccentricity min/max:')
print(df['e'].min(), df['e'].max())

print('\nAbsolute magnitude H min/max:')
print(df['H'].min(), df['H'].max())

Diameter (km) min/max:
0.0009129793677462 37.675

MOID (AU) min/max:
4.54e-07 0.708

Eccentricity min/max:
0.0028 0.9964

Absolute magnitude H min/max:
9.17 32.95


In [9]:
# Checking if any asteroids have a diameter less than or equal to 0
print('Asteroids with diameter_km <= 0:', (df['diameter_km'] <= 0).sum())

Asteroids with diameter_km <= 0: 0


In [10]:
# Checking to see if there are any duplicated rows
print('Number of duplicate rows:', df.duplicated().sum())

Number of duplicate rows: 0


In [11]:
# Start looking at summary statistics
df.describe()

,spkid,H,diameter_km,diameter_m,e,a,i,q,ad,per,per_y,moid_au,moid_km,moid_lunar_distances,n,condition_code,data_arc,data_arc_years
count,4.074200e+04,40742.000000,40742.000000,40742.000000,40742.000000,40742.000000,40742.000000,40742.000000,40742.000000,4.074200e+04,40742.000000,4.074200e+04,4.074200e+04,40742.000000,40742.000000,40742.000000,40742.000000,40742.000000
mean,2.843672e+07,23.659815,0.167436,167.433037,0.434117,1.754508,11.873518,0.915020,2.593992,9.560965e+02,2.617955,8.287206e-02,1.239748e+07,32.251299,0.529971,5.224903,1409.013966,3.857719
std,2.426461e+07,2.833311,0.405987,405.988101,0.176909,2.010194,10.560935,0.222079,4.001222,1.231075e+04,33.777900,9.653336e-02,1.444119e+07,37.567878,0.284421,3.170836,3103.040435,8.495631
min,3.002856e+06,9.170000,0.000913,0.900000,0.002800,0.461800,0.010000,0.069000,0.650000,1.150000e+02,0.314000,4.540000e-07,6.800000e+01,0.000000,0.000150,0.000000,1.000000,0.000000
25%,3.745033e+06,21.630000,0.025613,25.600000,0.300100,1.287250,4.380000,0.793250,1.650000,5.340000e+02,1.460000,1.200000e-02,1.795174e+06,4.670000,0.309200,2.000000,6.000000,0.020000
50%,2.045784e+07,24.000000,0.056294,56.300000,0.448100,1.681000,8.430000,0.963000,2.410000,7.960000e+02,2.180000,4.310000e-02,6.447668e+06,16.770000,0.452150,7.000000,22.000000,0.060000
75%,5.433118e+07,25.710000,0.168446,168.400000,0.562775,2.166000,16.620000,1.058000,3.340000,1.160000e+03,3.190000,1.220000e-01,1.825094e+07,47.480000,0.674675,8.000000,901.500000,2.467500
max,5.460683e+07,32.950000,37.675000,37675.000000,0.996400,350.300000,165.600000,1.300000,699.320000,2.390000e+06,6560.000000,7.080000e-01,1.059153e+08,275.530000,3.141000,9.000000,46582.000000,127.530000


In [13]:
df.describe(include='str')

,full_name,pdes,size_category,class,first_obs,last_obs
count,40742,40742,40742,40742,40742,40742
unique,40742,40742,4,4,7763,6144
top,433 Eros (A898 PA),433,Small (25-140m) — Local damage,APO,2014-04-23,2026-03-22
freq,1,1,19077,23062,73,115


In [15]:
df['pha'].value_counts(normalize=True)

pha
False    0.937681
True     0.062319
Name: proportion, dtype: float64

In [16]:
df['class'].value_counts()

class
APO    23062
AMO    14343
ATE     3299
IEO       38
Name: count, dtype: int64

In [17]:
# Compare key features between PHA's and non-PHA's
df.groupby('pha')[['diameter_km', 'H', 'moid_au', 'e', 'a', 'i']].agg(['mean', 'median', 'min', 'max'])

diameter_km                                      H                       \
             mean    median       min     max       mean median    min    max   
pha                                                                             
False    0.151041  0.051105  0.000913  37.675  23.885413  24.21   9.17  32.95   
True     0.414120  0.274449  0.076000   7.000  20.265349  20.59  14.08  22.57   

        moid_au          ...       e                 a                         \
           mean  median  ...     min     max      mean median     min     max   
pha                      ...                                                    
False  0.086806  0.0489  ...  0.0028  0.9964  1.750247  1.673  0.4618  350.30   
True   0.023687  0.0234  ...  0.0122  0.9670  1.818608  1.804  0.6310   17.81   

               i                      
            mean median   min    max  
pha                                   
False  11.701265   8.32  0.01  165.6  
True   14.465325  10.30  0.15  134.0  

[2 rows x 24 columns]

## Key Insights from Summary Statistics

After exploring the basic statistics of the cleaned Near-Earth Asteroid dataset, a clear story begins to emerge.

### Most Near-Earth Asteroids are small and relatively harmless
The vast majority of the 40,742 asteroids in this dataset are small. The median diameter is only about 50 meters, and even the average stays well under 1 km. Only a small fraction reach sizes capable of causing regional or global damage.

### Getting close to Earth’s orbit is surprisingly common
More than half of all asteroids in the dataset have a Minimum Orbit Intersection Distance (MOID) below the 0.05 AU threshold used in the official definition of a Potentially Hazardous Asteroid. In other words, many asteroids *geometrically* come close to Earth’s path. However, closeness alone is not enough to make them dangerous.

### What actually makes an asteroid “Potentially Hazardous”?
When we directly compare PHAs versus non-PHAs, two factors stand out dramatically:

- **Size**: Potentially Hazardous Asteroids are significantly larger (median diameter ~274 m vs ~51 m for non-PHAs).
- **Proximity**: PHAs have much smaller MOIDs (median 0.023 AU vs 0.049 AU).

Absolute magnitude (H) strongly supports the size difference — PHAs are noticeably brighter on average, which is expected for larger objects.

Orbital shape (eccentricity) and inclination show only minor differences and do not appear to be strong predictors on their own.

### Orbit Class Distribution
Apollo (APO) asteroids dominate the dataset (over 56%), followed by Amor (AMO). These Earth-approaching and Earth-crossing populations are exactly where we would expect most hazardous objects to be found.

### Bottom line so far
Being labeled a Potentially Hazardous Asteroid is relatively rare (only 6.2% of the dataset). The combination of **sufficient size** and **sufficiently close orbital approach** is what separates the dangerous minority from the much larger population of small, distant, or poorly timed objects.

These insights give us a strong foundation for the next stage: visual exploration and, eventually, building a machine learning model that can identify the truly hazardous asteroids.